# ROSP-9: System availability / uptime

PTH Tier 1 metric **ROSP-9** - {{ params.start_dayobs }} to {{ params.end_dayobs }} (dayObs, inclusive)

- Jira epic: [ROSP-9](https://rubinobs.atlassian.net/browse/ROSP-9)
- Confluence (Hub page; no metric page yet): [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Erik Dennihy |
| Owner | Sandrine Thomas |
| Category | 1 - Observatory Operations |
| Layer | 2 |
| Target | 90% availability of observable nights; &le;14 days scheduled and &le;21 days unscheduled downtime per year |

Method version: v1.0 (draft)

Status: first implementation, **partial**. The availability fraction is computed; the scheduled / unscheduled split and the subsystem attribution are not yet possible (see [Results](#results)). Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501

## What this notebook measures

> Of the observable night time that was **not lost to weather**, what fraction of the time was the observatory **not in `FAULT` or `DOWNTIME`**?

$$\text{availability} = \frac{\text{observable night} - \text{weather} - (\text{fault or downtime outside weather})}{\text{observable night} - \text{weather}}$$

**Read this number together with time-loss accounting ([ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11)).** An availability figure on its own does not say *why* time was lost. Weather is removed from the denominator here precisely so that weather is not charged against the engineering systems.

## How the notebook is organised

The sections follow the same order as the question above:

1. **Data sources:** where the night boundaries and the observatory states come from.
2. **Selection:** which time counts. Start from the observable night, remove weather; what remains is the denominator.
3. **Computation:** inside that selected time, how much was the system available?
4. **Results:** the headline number against the 90% target, and a calendar showing it night by night.
5. **Configuration-change markers:** notes on changes to the system or the method that could explain steps in the series.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)

not_times_square = True

In [ ]:
# Validate the dayObs range and report it. Times Square substitutes the
# parameters above as Python ints, but coerce anyway so a string survives.
from datetime import datetime

start_dayobs = int(start_dayobs)
end_dayobs = int(end_dayobs)

if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )

start_date = datetime.strptime(str(start_dayobs), "%Y%m%d").date()
end_date = datetime.strptime(str(end_dayobs), "%Y%m%d").date()
n_nights = (end_date - start_date).days + 1

print(f"dayObs range: {start_dayobs} to {end_dayobs} (inclusive)")
print(f"Dates: {start_date.isoformat()} to {end_date.isoformat()} ({n_nights} nights)")

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights
# may be missing or stale; install/upgrade it there. When running
# anywhere else (e.g. an SDF login node) this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

## Data sources

Two inputs are needed, and they answer different questions.

| Input | Where it comes from |
|---|---|
| **Twilight Boundary** | Computed with `rubin_nights.dayobs_utils` (no database needed) |
| **Observatory State** | The EFD `observatoryStatus` topic, via `rubin_nights.observatory_status` |

### 1. The observable night

For each dayObs, the observable night runs from the moment the Sun is at altitude **-12 deg** in the evening to the moment it is at **-12 deg** in the morning. This is the definition chosen in [ROSP-22](https://rubinobs.atlassian.net/browse/ROSP-22) (notebook `observable_night_definition_ROSP-22`). It depends only on the Sun, so it is the same for every year and does not change with weather, faults or the dome. The definition must be held fixed: changing it later is a series break, not a restatement.

### 2. The observatory state

The observatory state is recorded in the EFD. The possible states are:

| State | Meaning for this notebook |
|---|---|
| `WEATHER` | Time lost to weather. **Removed from the denominator** (Selection). |
| `OPERATIONAL` | The observatory is available and observing. **Available.** |
| `IDLE` | The observatory is available but not observing. **Available.** |
| `UNKNOWN` | The state could not be determined. **Treated as available** (it is not recorded as a fault). |
| `FAULT` | A fault is active. **Not available.** |
| `DOWNTIME` | The observatory is in declared downtime. **Not available.** |

**Planned and unplanned downtime are not separated yet.** The EFD state `DOWNTIME` is a mix of scheduled (planned) and intentional but unplanned observatory shutdown. `FAULT` is treated as unscheduled downtime. Due to the ambiguous use of `DOWNTIME`, a clean separation between palnned and unplanned downtime cannot be made and they are therefore rolled together as "not available" until [ROSP-28](https://rubinobs.atlassian.net/browse/ROSP-28) is resolved.

The next cells load the tools, define the observable night, and query the states.

In [ ]:
# Imports, and the connection to the EFD.
# Imports live here (not in the parameters cell) because Times Square replaces the parameters cell wholesale.
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.observatory_status as observatory_status

# The Sun-position code needs the shared rubin_sim data. If it is not set, point at the shared copy at USDF.
if os.getenv("RUBIN_SIM_DATA_DIR") is None:
    for _data_dir in (
        "/sdf/data/rubin/shared/rubin_sim_data",
        "/sdf/group/rubin/shared/rubin_sim_data",
    ):
        if os.path.isdir(_data_dir):
            os.environ["RUBIN_SIM_DATA_DIR"] = _data_dir
            break

# On an RSP (including Times Square) the default endpoints and tokens are right; elsewhere get_clients() looks for
# your own USDF token (see rubin_nights.get_access_token).
if os.getenv("EXTERNAL_INSTANCE_URL", "") != "":
    tokenfile = None
    site = None
endpoints = connections.get_clients()

In [ ]:
# THE DEFINITION OF THE OBSERVABLE NIGHT (from ROSP-22): Sun at altitude -12 deg in the evening to -12 deg in the morning.
TWILIGHT_ALT = (
    -12.0
)  # Sun altitude (deg) at the start (evening) and end (morning) of the observable night


def observable_night(day_obs):
    """Start and end of the observable night of one dayObs, as astropy Times."""
    start, end = rn_dayobs.day_obs_sunset_sunrise(day_obs, TWILIGHT_ALT)
    return start, end

In [ ]:
# The list of nights to analyse: every dayObs from start_dayobs to end_dayobs, but not nights that have not happened yet.
today_dayobs = rn_dayobs.day_obs_str_to_int(rn_dayobs.today_day_obs())
last_dayobs = min(
    end_dayobs, today_dayobs
)  # a night in the future has no states to look at


def day_obs_range(first_day_obs, last_day_obs):
    """All dayObs (as YYYYMMDD integers) from first_day_obs to last_day_obs, inclusive."""
    days = pd.date_range(
        pd.to_datetime(str(first_day_obs), format="%Y%m%d"),
        pd.to_datetime(str(last_day_obs), format="%Y%m%d"),
    )
    return [int(d.strftime("%Y%m%d")) for d in days]


day_obs_list = day_obs_range(start_dayobs, last_dayobs)

# Times that bracket all of those nights, for the EFD query (dayObs starts at 12:00 UTC of its date).
t_start = rn_dayobs.day_obs_to_time(start_dayobs)
t_end = rn_dayobs.day_obs_to_time(last_dayobs) + TimeDelta(1, format="jd")
print(f"{len(day_obs_list)} nights, dayObs {start_dayobs} to {last_dayobs}")
if last_dayobs < end_dayobs:
    print(
        f"(end_dayobs {end_dayobs} is in the future; the analysis stops at {last_dayobs})"
    )

### Querying the observatory states

`get_observatory_state_times` returns the EFD states as **periods**: one row per stretch of time (`start`, `end`) during which a state (`type`) was in force, together with the night it belongs to (`day_obs`) and that night's twilight times (`sunset12`, `sunrise12`).

**These periods overlap.** A `WEATHER` period typically spans the whole night while `OPERATIONAL`, `IDLE` or `FAULT` periods run inside it, so the hours of the states do *not* add up to the length of the night. We therefore never add per-state hours together. Instead, later cells take the **union** of the periods of interest (the set of moments covered by at least one of them), which counts every moment once.

In [ ]:
# Query the EFD for the state periods of all nights in the range.
obs_status_periods = observatory_status.get_observatory_state_times(
    t_start, t_end, endpoints["efd"]
)
print(
    f"{len(obs_status_periods)} state periods on {obs_status_periods.day_obs.nunique()} of {len(day_obs_list)} nights"
)
display(obs_status_periods.head(10))

### Two corrections to the raw state record

Both corrections were developed in ROSP-22, and both are deliberately visible here rather than hidden in a library call.

1. **Nights with no status messages are carried forward.** The EFD logs `observatoryStatus` only when the state *changes*. A night with no messages therefore means the previous state persisted (for example the observatory sat in `DOWNTIME` for weeks). The library builds periods only for nights that have a message, so these quiet nights would silently vanish from the statistics, and the availability would look better than it was. `fill_missing_nights` repeats the state in force at sunrise of the last known night over each quiet night. *This is an assumption:* a long gap could also be a hole in the EFD record. Nights before the first message of the range cannot be filled and are left out.
2. **`NIGHT_STATE_OVERRIDES` replaces the logged state of whole nights.** It is a deliberate manual choice, applied only to the nights listed, never as a general rule. **Currently every night from 15 July 2026 (`OVERRIDE_FROM_DAYOBS = 20260715`) on is set to `WEATHER | DOWNTIME`**, whatever the EFD logged. This supersedes the earlier override of `20260925` alone (logged `UNKNOWN`, actually `DOWNTIME`), which lies inside that range. The overridden state is also what gets carried forward over the quiet nights that follow. Because weather is removed from the selection, **this makes every one of those nights count as lost to weather, and none of their downtime is charged to the system** (see Checks on the computation).

Every adjustment is printed when it is applied.

In [ ]:
# Manual overrides of the state for a whole night, as {dayObs: types in force all night}.
# From 15 July 2026 on, the whole time is treated as WEATHER | DOWNTIME. (This replaces the earlier single-night override of
# 20260925, which lies inside this range.) Nights with no status messages after the last overridden night are then
# carried forward as WEATHER | DOWNTIME by fill_missing_nights below.
OVERRIDE_FROM_DAYOBS = 20260715
NIGHT_STATE_OVERRIDES = {
    d: ["WEATHER", "DOWNTIME"] for d in day_obs_range(OVERRIDE_FROM_DAYOBS, last_dayobs)
}


def override_night_states(periods, overrides):
    """Replace all periods of the listed nights by one full-night period per type."""
    changed = []
    for d, types in overrides.items():
        night = periods[periods.day_obs == d]
        if len(night) == 0:
            continue  # no periods that night (not in this range, or a quiet night that is carried forward below)
        sunset12, sunrise12 = night.sunset12.iloc[0], night.sunrise12.iloc[0]
        hours = (sunrise12 - sunset12).total_seconds() / 3600
        new = pd.DataFrame(
            [
                {
                    "day_obs": d,
                    "sunset12": sunset12,
                    "sunrise12": sunrise12,
                    "start": sunset12,
                    "end": sunrise12,
                    "hours": hours,
                    "type": t,
                }
                for t in types
            ]
        )
        periods = pd.concat([periods[periods.day_obs != d], new], ignore_index=True)
        changed.append(d)
    if changed:
        print(
            f"override: {len(changed)} nights ({changed[0]} to {changed[-1]}) with logged states replaced by "
            f"{sorted(set(t for ts in overrides.values() for t in ts))}"
        )
    return periods.sort_values(["day_obs", "start"], ignore_index=True)


def fill_missing_nights(periods, nights, tol_s=1.0):
    """Carry the last known state forward across nights with no EFD status messages.

    The types still in force at sunrise of the last night with periods (e.g. DOWNTIME) are repeated over each
    following empty night (clipped to now; a night that has not started is skipped).
    Returns (periods, filled_nights, unfilled_nights, carried_types).
    """
    have = set(periods.day_obs.unique())
    tnow = Time.now().utc
    last_types, rows, filled, unfilled = None, [], [], []
    for d in sorted(nights):
        if d in have:
            per = periods[periods.day_obs == d]
            in_force = per[(per.sunrise12 - per.end).dt.total_seconds() <= tol_s]
            last_types = sorted(in_force.type.unique())
            continue
        sunset, sunrise = (
            t.utc for t in rn_dayobs.day_obs_sunset_sunrise(d, TWILIGHT_ALT)
        )
        if tnow <= sunset:
            continue  # night has not started yet
        if last_types is None:
            unfilled.append(d)  # nothing earlier in the range to carry forward
            continue
        sunrise = min(sunrise, tnow)
        hours = (sunrise - sunset).jd * 24
        for typ in last_types:
            rows.append(
                {
                    "day_obs": d,
                    "sunset12": sunset.datetime,
                    "sunrise12": sunrise.datetime,
                    "start": sunset.datetime,
                    "end": sunrise.datetime,
                    "hours": hours,
                    "type": typ,
                }
            )
        filled.append(d)
    if rows:
        periods = pd.concat(
            [periods, pd.DataFrame(rows)], ignore_index=True
        ).sort_values(["day_obs", "start"], ignore_index=True)
    return periods, filled, unfilled, last_types


filled_nights, unfilled_nights = [], []
if len(obs_status_periods) > 0:
    obs_status_periods = override_night_states(
        obs_status_periods, NIGHT_STATE_OVERRIDES
    )
    obs_status_periods, filled_nights, unfilled_nights, carried_types = (
        fill_missing_nights(obs_status_periods, day_obs_list)
    )
    if filled_nights:
        print(
            f"{len(filled_nights)} nights with no status messages carried forward as {carried_types}: "
            f"{filled_nights[0]} to {filled_nights[-1]}"
        )
    if unfilled_nights:
        print(
            f"{len(unfilled_nights)} nights before the first status message left out: {unfilled_nights}"
        )
else:
    print("No observatory state periods found in the EFD for this range.")

## Selection

Selection decides **which time counts** towards the availability. It is the denominator, and it determines the answer at least as much as the numerator does, so it is done in two explicit steps:

1. **Start from the observable night.** For every dayObs, the time between the Sun reaching -12 deg in the evening and -12 deg in the morning (Data sources, above).
2. **Remove the time lost to weather.** Time in the `WEATHER` state is taken out. Weather is outside the control of the observatory's systems, and leaving it in would charge weather losses against engineering.

What is left, *observable night minus weather*, is the **selected time**. The availability is measured only within it.

> Weather is removed here, for this metric, because the question is about the *system*. This does not change the definition of the observable night itself (ROSP-22), which is the Sun alone.

In [ ]:
# STEP 1: the observable night of every night in the range.
# The state periods carry each night's -12 deg twilight times (sunset12, sunrise12), which are the same Sun-only
# definition as above. We take them from there so the night window and the state periods use identical timestamps.
nights = obs_status_periods.groupby("day_obs")[["sunset12", "sunrise12"]].first()
nights = nights.rename(columns={"sunset12": "night_start", "sunrise12": "night_end"})
nights["night_hours"] = (
    nights.night_end - nights.night_start
).dt.total_seconds() / 3600

# Cross-check against the function that defines the observable night, on the nights that have completely finished
# (carried-forward nights are clipped at the current time, so a night in progress is shorter).
complete = nights[nights.night_end < pd.Timestamp.utcnow().tz_localize(None)]
sun_hours = pd.Series(
    {
        d: (lambda s_e: (s_e[1] - s_e[0]).jd * 24)(observable_night(d))
        for d in complete.index
    }
)
print(
    f"{len(nights)} nights with state data; longest difference between the state-table night and the "
    f"Sun definition: {(complete.night_hours - sun_hours).abs().max() * 60:.2f} minutes"
)
display(nights.round(2).head())

In [ ]:
# A helper used from here on: how many hours of a night are covered by periods of the given state types?
# Periods can overlap, so this takes the UNION: it walks the periods in time order and counts each moment once.
def union_hours(periods, types, lo, hi):
    """Hours covered by the union of `types` periods, clipped to the window [lo, hi]."""
    d = periods[periods.type.isin(types)].sort_values("start")
    total, cur_end = 0.0, lo
    for s, e in zip(d.start, d.end):
        s, e = max(s, lo, cur_end), min(e, hi)
        if e > s:
            total += (e - s).total_seconds() / 3600
            cur_end = e
    return total


# Group the periods by night once, so each night's periods are easy to pick out.
periods_by_night = {d: g for d, g in obs_status_periods.groupby("day_obs")}

In [ ]:
# STEP 2: remove the time lost to weather. selected time = observable night - weather.
nights["weather_hours"] = [
    union_hours(periods_by_night[d], ["WEATHER"], w.night_start, w.night_end)
    for d, w in nights.iterrows()
]
nights["selected_hours"] = nights.night_hours - nights.weather_hours

display(nights[["night_hours", "weather_hours", "selected_hours"]].round(2).head(10))

In [ ]:
# What the selection kept, over the whole range.
total_night = nights.night_hours.sum()
total_weather = nights.weather_hours.sum()
total_selected = nights.selected_hours.sum()
n_weathered_out = int((nights.selected_hours < 0.05).sum())

print(f"Observable night:       {total_night:8.1f} h over {len(nights)} nights")
print(
    f"Lost to weather:        {total_weather:8.1f} h ({total_weather / total_night:.1%} of the observable night)"
)
print(
    f"Selected (night - wx):  {total_selected:8.1f} h ({total_selected / total_night:.1%} of the observable night)"
)
print(f"Nights with no selected time (entirely weather): {n_weathered_out}")

## Computation

Now we ask, **inside the selected time**, how much of it the system was available.

The rule is the one from the Data sources table:

- **Available:** the observatory is in `OPERATIONAL`, `IDLE` or `UNKNOWN`.
- **Not available:** the observatory is in `FAULT` or `DOWNTIME` (rolled together; planned and unplanned cannot be separated yet, see [ROSP-28](https://rubinobs.atlassian.net/browse/ROSP-28)).

Because the periods overlap, we count the *not available* time and subtract it, instead of adding up the available states:

$$\text{available} = \text{selected} - \text{not available},\qquad \text{availability} = \frac{\text{available}}{\text{selected}}$$

**How to count "not available outside weather".** We need the hours that are in `FAULT` or `DOWNTIME` *and not* in `WEATHER`. With union lengths $U(\cdot)$, this is

$$\bigl|(F \cup D) \setminus W\bigr| \;=\; U(F \cup D \cup W) - U(W)$$

that is, add the fault and downtime periods to the weather periods, and subtract the weather alone. Whatever is left was fault or downtime that happened while the weather was fine. Fault or downtime that overlaps weather is not charged to the system.

The cells below do this one step at a time, per night, then pool the nights into the final fraction.

In [ ]:
# The states that count as available and as not available. Change them here and nothing else needs to change.
AVAILABLE_STATES = ["OPERATIONAL", "IDLE", "UNKNOWN"]
NOT_AVAILABLE_STATES = [
    "FAULT",
    "DOWNTIME",
]  # planned and unplanned are rolled together (ROSP-28)
WEATHER_STATES = ["WEATHER"]

print(f"available:     {AVAILABLE_STATES}")
print(
    f"not available: {NOT_AVAILABLE_STATES}  (rolled together: planned vs unplanned is not yet possible)"
)
print(f"removed first: {WEATHER_STATES}")

In [ ]:
# Per night: hours that were fault/downtime AND outside weather = U(fault, downtime, weather) - U(weather).
not_available = []
for d, w in nights.iterrows():
    per = periods_by_night[d]
    u_all = union_hours(
        per, NOT_AVAILABLE_STATES + WEATHER_STATES, w.night_start, w.night_end
    )
    not_available.append(
        u_all - w.weather_hours
    )  # w.weather_hours is U(weather) from Selection
nights["not_available_hours"] = np.clip(
    not_available, 0, None
)  # clip only removes rounding noise

# Available = selected - not available, and the fraction of the selected time that was available.
nights["available_hours"] = nights.selected_hours - nights.not_available_hours
nights["availability"] = np.where(
    nights.selected_hours
    > 0.05,  # a night with no selected time (all weather) has no availability
    nights.available_hours / nights.selected_hours,
    np.nan,
)

display(
    nights[
        [
            "night_hours",
            "weather_hours",
            "selected_hours",
            "not_available_hours",
            "available_hours",
            "availability",
        ]
    ]
    .round(2)
    .head(10)
)

In [ ]:
# THE COMPUTATION: pool all nights. Total available hours over total selected hours (so a long night counts more
# than a short one, and nights with no selected time contribute nothing).
available_hours_total = nights.available_hours.sum()
selected_hours_total = nights.selected_hours.sum()
not_available_hours_total = nights.not_available_hours.sum()
availability = available_hours_total / selected_hours_total

print(f"selected time:           {selected_hours_total:8.1f} h")
print(f"not available (F or D):  {not_available_hours_total:8.1f} h")
print(f"available:               {available_hours_total:8.1f} h")
print(f"availability:            {availability:8.1%}")

### Checks on the computation

Two diagnostics, not part of the metric, to show how much the choices above matter:

- **Fault or downtime hidden by weather.** Weather is removed first, so any `FAULT` or `DOWNTIME` that overlaps `WEATHER` is not charged to the system. This is intended (weather is not the engineers' doing), but if the recorded weather state persists through a long downtime, a lot of downtime can disappear from the metric this way. The size of that hidden time is shown below so it is never a surprise.
  - **Watch for carried-forward weather.** The carry-forward correction repeats whatever state was in force at the end of the last night with messages. If that state included `WEATHER` (as the `WEATHER | DOWNTIME` state logged in July did), the quiet nights are filled as weather *and* downtime, the whole night is removed by the selection, and the downtime on those nights is not counted against availability. Compare the hidden hours with the hours charged before trusting the headline number.
- **Time with no recorded state.** The first night of a range can begin before the first status message, leaving time in which the state is unknown to the EFD record. Such time counts as available in the formula above (it is not recorded as fault or downtime), so its size is shown.

In [ ]:
# Diagnostic 1: hours of FAULT/DOWNTIME that overlap WEATHER (removed by the selection, not charged to the system).
# |(F or D) and W| = U(F or D) + U(W) - U(F or D or W)
hidden = []
unrecorded = []
for d, w in nights.iterrows():
    per = periods_by_night[d]
    u_fd = union_hours(per, NOT_AVAILABLE_STATES, w.night_start, w.night_end)
    u_all_states = union_hours(
        per,
        AVAILABLE_STATES + NOT_AVAILABLE_STATES + WEATHER_STATES,
        w.night_start,
        w.night_end,
    )
    hidden.append(
        u_fd
        + w.weather_hours
        - union_hours(
            per, NOT_AVAILABLE_STATES + WEATHER_STATES, w.night_start, w.night_end
        )
    )
    unrecorded.append(
        w.night_hours - u_all_states
    )  # Diagnostic 2: night time covered by no state at all
nights["fault_downtime_in_weather_hours"] = np.clip(hidden, 0, None)
nights["no_state_hours"] = np.clip(unrecorded, 0, None)

hidden_total = nights.fault_downtime_in_weather_hours.sum()
n_hidden_nights = int((nights.fault_downtime_in_weather_hours > 0.05).sum())
print(
    f"Fault/downtime overlapping weather (not charged): {hidden_total:.1f} h on {n_hidden_nights} nights "
    f"(vs {not_available_hours_total:.1f} h charged)"
)
print(
    f"Night time with no recorded state: {nights.no_state_hours.sum():.1f} h "
    f"on {int((nights.no_state_hours > 0.05).sum())} nights (counted as available)"
)

## Results

The result is reported two ways: a **headline statistic** compared with the 90% target, and a **calendar** showing the availability of every night.

### What is reported, and what cannot be yet

The metric definition asks for four things each cycle. Only the first can be reported today.

| Reported each cycle | Status |
|---|---|
| Availability against 90%, with the observable-night definition stated | **Reported below.** Observable night = Sun -12 to -12 deg (ROSP-22), minus weather. |
| Scheduled downtime days against the 14-day allocation | **Not possible yet.** The EFD state does not distinguish scheduled from unscheduled downtime. Pending [ROSP-28](https://rubinobs.atlassian.net/browse/ROSP-28). |
| Unscheduled downtime days against the 21-day allocation | **Not possible yet**, for the same reason (ROSP-28). |
| Subsystem attribution of unscheduled downtime | **Not possible yet.** It needs the cause breakdown from time-loss accounting, pending [ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24) and [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11). |

Because planned and unplanned time cannot be separated, **`FAULT` and `DOWNTIME` are rolled together** in the number below. A scheduled maintenance window therefore counts against the availability exactly like an unplanned failure. The true availability against unscheduled loss alone would be higher.

In [ ]:
# HEADLINE RESULT: availability against the 90% target, shown as a highlighted box so it stands out from the other output.
from IPython.display import HTML, display

AVAILABILITY_TARGET = 0.90  # target quoted in the metric definition: 90% availability of observable nights

meets = availability >= AVAILABILITY_TARGET
verdict = "MEETS" if meets else "DOES NOT MEET"
accent = (
    "#1a7f37" if meets else "#c62828"
)  # the verdict is also written out, so colour is not the only cue

hidden_note = (
    f"<li>{hidden_total:.1f} h of FAULT/DOWNTIME overlapped WEATHER and was <b>not charged</b> to the system (see Checks on the computation).</li>"
    if hidden_total > 0.05
    else ""
)

display(
    HTML(
        f"""
<div style="border: 3px solid {accent}; border-radius: 8px; padding: 14px 20px; margin: 10px 0; max-width: 900px;
            background: rgba(128, 128, 128, 0.08); font-family: sans-serif;">
  <div style="font-size: 1.05em; font-weight: bold; letter-spacing: 0.04em;">
    ROSP-9 SYSTEM AVAILABILITY &mdash; dayObs {nights.index.min()} to {nights.index.max()} ({len(nights)} nights)
  </div>
  <div style="font-size: 2.6em; font-weight: bold; color: {accent}; margin: 6px 0 0 0;">{availability:.1%}</div>
  <div style="font-size: 1.2em; font-weight: bold; color: {accent};">
    {verdict} the {AVAILABILITY_TARGET:.0%} target &nbsp;({(availability - AVAILABILITY_TARGET) * 100:+.1f} percentage points)
  </div>
  <table style="margin: 10px 0; border-collapse: collapse;">
    <tr><td style="padding: 2px 16px 2px 0;">Observable night definition</td>
        <td><b>Sun {TWILIGHT_ALT:g} to {TWILIGHT_ALT:g} deg, minus weather</b></td></tr>
    <tr><td style="padding: 2px 16px 2px 0;">Selected time</td>
        <td><b>{selected_hours_total:.1f} h</b> ({total_weather:.1f} h of weather removed from {total_night:.1f} h of night)</td></tr>
    <tr><td style="padding: 2px 16px 2px 0;">Not available</td>
        <td><b>{not_available_hours_total:.1f} h</b> (FAULT + DOWNTIME rolled together)</td></tr>
  </table>
  <div style="font-weight: bold;">Caveats</div>
  <ul style="margin: 2px 0 0 0;">
    <li>Planned and unplanned downtime are <b>not separated</b> (pending ROSP-28), so scheduled work counts against the availability.</li>
    <li>Subsystem attribution is pending ROSP-24 and ROSP-11.</li>
    {hidden_note}
  </ul>
</div>
"""
    )
)

### Night-by-night calendar

The headline number hides how the availability is distributed. The calendar shows it for every night: one row per week (Monday to Sunday), one cell per night, the **percentage of the selected time that was available** written in the cell.

The colour scale is centred on the **90% target**:

- **Redder** the further a night falls *below* 90%, and **greener** the further it rises *above* 90%. A night close to 90% is pale grey.
- Both arms get darker with distance from the target, so the scale still reads if red and green look alike to you; the number in every cell says the same thing.
- Grey `wx` cells are nights with **no selected time** at all because the whole night was lost to weather, so they have no availability to show.

This is the same calendar layout as in the ROSP-22 observable-night notebook.

In [ ]:
# Helpers for the calendar plot (adapted from the ROSP-22 notebook; notebooks here must not import from each other).
from matplotlib.colors import LinearSegmentedColormap, Normalize

# Diverging colour scale centred on the 90% target (the neutral grey sits at exactly 90): reds below it, greens above it.
# Position = availability / 100 on a Normalize(0, 100) scale, so the red arm spans 0-90% and the green arm 90-100%.
# Both arms get darker away from the target, so lightness as well as hue tells how far a night is from 90%.
AVAILABILITY_CMAP = LinearSegmentedColormap.from_list(
    "availability_vs_target",
    [
        (0.00, "#7a1414"),  # 0%: dark red
        (0.45, "#d03b3b"),  # 45%: red
        (0.90, "#f0efec"),  # 90%: neutral grey, the target
        (0.95, "#0ca30c"),  # 95%: green
        (1.00, "#066306"),  # 100%: dark green
    ],
)


def calendar_grid(values):
    """Weeks x 7 array (Monday to Sunday) of a per-night value (Series indexed by dayObs), and the first Monday."""
    days = pd.to_datetime(values.index.astype(str), format="%Y%m%d")
    first_monday = days.min() - pd.Timedelta(days=days.min().weekday())
    nweeks = (days.max() - first_monday).days // 7 + 1
    cal = np.full((nweeks, 7), np.nan)
    for day, v in zip(days, values.values):
        cal[(day - first_monday).days // 7, day.weekday()] = v
    return cal, first_monday


def draw_calendar(
    ax,
    values,
    cmap,
    norm,
    title,
    label=lambda v: f"{v:.0f}",
    ytick_every=4,
    fontsize=6.5,
):
    """Draw a calendar of a per-night value on `ax`, with the value written in every cell."""
    cal, first_monday = calendar_grid(values)
    nweeks = cal.shape[0]
    im = ax.imshow(np.ma.masked_invalid(cal), cmap=cmap, norm=norm, aspect="auto")
    for (i, j), v in np.ndenumerate(cal):
        if not np.isnan(v):
            r, g, b, _ = cmap(
                norm(v)
            )  # white text on dark cells, dark text on light cells
            dark = 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.4
            ax.text(
                j,
                i,
                label(v),
                ha="center",
                va="center",
                fontsize=fontsize,
                color="white" if dark else "#0b0b0b",
            )
    ax.set_xticks(range(7))
    ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=8)
    ax.xaxis.tick_top()
    ticks = range(0, nweeks, ytick_every)
    ax.set_yticks(ticks)
    ax.set_yticklabels(
        [(first_monday + pd.Timedelta(weeks=int(i))).strftime("%b %d") for i in ticks],
        fontsize=8,
    )
    ax.set_xticks(np.arange(-0.5, 7, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nweeks, 1), minor=True)
    ax.grid(which="minor", color="white", lw=1.5)
    ax.tick_params(which="both", length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, loc="left", fontsize=10, pad=24)
    return im

In [ ]:
# Calendar of the availability of each night, in percent of the selected time.
# Nights lost entirely to weather have no availability; give them -1 so they draw in the "under" colour (grey).
cal_values = (nights.availability * 100).fillna(-1)

cal_cmap = AVAILABILITY_CMAP.copy()
cal_cmap.set_under(
    "#c3c2b7"
)  # whole night lost to weather (mid grey, distinct from the near-white 90% midpoint)
cal_cmap.set_bad("white")  # not a night in the range

fig, ax = plt.subplots(figsize=(6, 14))
im = draw_calendar(
    ax,
    cal_values,
    cal_cmap,
    Normalize(0, 100),
    f"System availability per night (% of observable night minus weather)\n"
    f"overall {availability:.1%} vs target {AVAILABILITY_TARGET:.0%}; FAULT and DOWNTIME rolled together",
    label=lambda v: "wx" if v < 0 else f"{v:.0f}",
    ytick_every=2,
)
cbar = fig.colorbar(
    im,
    ax=ax,
    orientation="horizontal",
    fraction=0.025,
    pad=0.04,
    aspect=30,
    extend="min",
)
cbar.set_ticks([0, 25, 50, 75, AVAILABILITY_TARGET * 100, 100])
cbar.set_label(
    "Availability (%): red below the 90% target, green above\ngrey wx = whole night lost to weather",
    fontsize=9,
)

## Configuration-change markers

This section records **changes that can move the availability series for reasons other than how well the system performed**, so that a step in the numbers is not mistaken for a trend. There are two kinds:

- **Method changes**: a different observable-night definition, a different set of available / not-available states, a different treatment of weather or of the state record. By the PTH rules, a change of definition is a **series break**, not a restatement. The current method is **v1.0 (draft)**.
- **System changes**: software releases, hardware changes or reclassification of states (for example, once ROSP-28 splits planned from unplanned downtime) that change what the EFD states mean.

### Current method decisions (v1.0 draft)

| Decision | Setting in this notebook | Status |
|---|---|---|
| Observable night | Sun -12 to -12 deg (`TWILIGHT_ALT`), from ROSP-22 | Decided, held fixed |
| Weather | Removed from the denominator (`WEATHER_STATES`) | Decided for this metric |
| Available / not available | `OPERATIONAL`, `IDLE`, `UNKNOWN` / `FAULT`, `DOWNTIME` rolled together (`AVAILABLE_STATES`, `NOT_AVAILABLE_STATES`) | Subject to change once ROSP-28 separates planned from unplanned |
| **State from 15 July 2026 on** | **Every night is set to `WEATHER \| DOWNTIME` (`OVERRIDE_FROM_DAYOBS = 20260715`, `NIGHT_STATE_OVERRIDES`)** | **Subject to change** |

**The 15 July 2026 override.** From 15 July 2026 on, the whole time is treated as `WEATHER | DOWNTIME`, whatever the EFD logged. Because weather is removed first, those nights drop out of the selection, so their downtime is not charged to the system, and the availability reflects only the nights before 15 July (about 16 nights, 29 June to 14 July). **This is subject to change.** If it changes, that is a change of method and a **series break**: record the new setting here with its date, and do not restate the earlier results as though they had used it. To reverse or alter it, change `OVERRIDE_FROM_DAYOBS` (or remove `NIGHT_STATE_OVERRIDES`) in the corrections cell and re-run.

The dictionaries below hold the changes as `{dayObs: description}`. `METHOD_CHANGES` is filled; `CONFIG_CHANGES` (system changes) is **empty** until the stewards agree what to record.

In [ ]:
# Method changes and system changes that affect how the availability series should be read: {dayObs: description}.
# Add new entries (with their date) when a decision changes; a change of method is a series break.
METHOD_CHANGES = {
    OVERRIDE_FROM_DAYOBS: (
        "Subject to change: every night from this dayObs on is set to WEATHER | DOWNTIME, "
        "so these nights are removed as weather and their downtime is not charged to the system."
    ),
}
CONFIG_CHANGES = (
    {}
)  # system / configuration changes (software, hardware, state reclassification): none recorded yet

for title, changes in [
    ("Method changes", METHOD_CHANGES),
    ("System changes", CONFIG_CHANGES),
]:
    in_range = {d: txt for d, txt in changes.items() if d <= nights.index.max()}
    print(f"{title}:")
    if in_range:
        for d, txt in sorted(in_range.items()):
            print(f"  {d}: {txt}")
    else:
        print("  none recorded for this range")